# Notebook 4: Feature Engineering

**Automotive Car Price Prediction Pipeline**

---

This notebook creates new features and removes redundant columns based on EDA findings.

**Input:** `workspace.default.cleaned_data` (from Notebook 2)

**Output:** `workspace.default.carprice_project_engineered_data`

## 1. Load Data from Catalog

In [0]:
import pandas as pd
import numpy as np

# Read from catalog into pandas
df = spark.table("workspace.default.cleaned_data").toPandas()
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Shape: (56133, 12)
Columns: ['make', 'model', 'priceUSD', 'year', 'condition', 'mileage_kilometers', 'fuel_type', 'volume_cm3', 'color', 'transmission', 'drive_unit', 'segment']


In [0]:
df.head()

,make,model,priceUSD,year,condition,mileage_kilometers,fuel_type,volume_cm3,color,transmission,drive_unit,segment
0,nissan,qashqai,16800,2016,with mileage,44000.0,petrol,2000.0,white,auto,front-wheel drive,j
1,nissan,qashqai,16900,2016,with mileage,80000.0,petrol,1200.0,red,auto,front-wheel drive,j
2,nissan,qashqai,9650,2010,with mileage,198000.0,petrol,1600.0,brown,mechanics,front-wheel drive,j
3,nissan,qashqai,9800,2011,with mileage,117000.0,petrol,1600.0,black,mechanics,front-wheel drive,j
4,nissan,qashqai,11500,2011,with mileage,93000.0,petrol,2000.0,other,auto,part-time four-wheel drive,j


## 2. Create New Features

In [0]:
# Feature 1: car_age = 2019 - year
# The dataset was collected in early 2019, so car_age represents age at time of listing
df['car_age'] = 2019 - df['year']

print("car_age created (2019 - year):")
print(df[['year', 'car_age']].head())

car_age created (2019 - year):
   year  car_age
0  2016        3
1  2016        3
2  2010        9
3  2011        8
4  2011        8


In [0]:
# Feature 2: mileage_per_year = mileage_km / max(car_age, 1)
# Average annual usage — indicates how heavily the car was driven
df['mileage_per_year'] = (df['mileage_kilometers'] / df['car_age'].clip(lower=1)).round(2)

print("mileage_per_year created:")
print(df[['mileage_kilometers', 'car_age', 'mileage_per_year']].head())

mileage_per_year created:
   mileage_kilometers  car_age  mileage_per_year
0             44000.0        3          14666.67
1             80000.0        3          26666.67
2            198000.0        9          22000.00
3            117000.0        8          14625.00
4             93000.0        8          11625.00


In [0]:
# Feature 3: make_category
# Group rare makes (< 100 cars) into 'other' to reduce cardinality
make_counts = df['make'].value_counts()
common_makes = make_counts[make_counts >= 100].index.tolist()
rare_makes = make_counts[make_counts < 100].index.tolist()

print(f"Common makes (>= 100 cars): {len(common_makes)}")
print(f"Rare makes (< 100 cars): {len(rare_makes)} — will be grouped as 'other'")

df['make_category'] = df['make'].apply(lambda x: x if x in common_makes else 'other')

print(f"\nmake_category distribution:")
print(df['make_category'].value_counts().to_string())

Common makes (>= 100 cars): 38
Rare makes (< 100 cars): 58 — will be grouped as 'other'

make_category distribution:
volkswagen       6849
audi             4014
bmw              4005
opel             3767
renault          3704
mercedes-benz    3537
ford             3072
peugeot          2868
nissan           2226
toyota           2176
mazda            2004
citroen          1988
hyundai          1499
other            1429
mitsubishi       1347
volvo            1231
kia              1196
skoda            1142
lada-vaz          999
honda             948
fiat              934
chevrolet         595
chrysler          516
seat              473
subaru            374
lexus             358
land-rover        347
suzuki            330
daewoo            316
rover             303
infiniti          247
alfa-romeo        227
gaz               224
jeep              191
porsche           175
ssangyong         139
uaz               137
saab              129
lancia            117


## 3. Create Price Category (for Stratified Split)

This column bins `price` into 3 categories so Notebook 5 can do a stratified train/test split, ensuring both sets have a similar price distribution.

> This column is used **only for splitting** and will be dropped before training.

In [0]:
# Bin price into 3 categories using percentiles (33rd and 66th)
p33 = df['priceUSD'].quantile(0.33)
p66 = df['priceUSD'].quantile(0.66)

def assign_price_category(price):
    if price <= p33:
        return 'budget'
    elif price <= p66:
        return 'mid'
    else:
        return 'luxury'

df['price_category'] = df['priceUSD'].apply(assign_price_category)

print(f"Price category bins:")
print(f"  budget  : price <= ${p33:,.0f}")
print(f"  mid     : ${p33:,.0f} < price <= ${p66:,.0f}")
print(f"  luxury  : price > ${p66:,.0f}")
print(f"\nPrice category distribution:")
print(df['price_category'].value_counts().to_string())

Price category bins:
  budget  : price <= $3,200
  mid     : $3,200 < price <= $7,900
  luxury  : price > $7,900

Price category distribution:
luxury    18905
mid       18659
budget    18569


## 4. Drop Redundant / High-Cardinality Columns

In [0]:
# Drop columns:
# - model: very high cardinality — too sparse for encoding
# - year: replaced by car_age
# - make: replaced by make_category
cols_to_drop = ['model', 'year', 'make']
df = df.drop(columns=cols_to_drop)

print(f"Dropped columns: {cols_to_drop}")
print(f"\nRemaining columns ({len(df.columns)}):")
for col in df.columns:
    print(f"  {col}")

Dropped columns: ['model', 'year', 'make']

Remaining columns (13):
  priceUSD
  condition
  mileage_kilometers
  fuel_type
  volume_cm3
  color
  transmission
  drive_unit
  segment
  car_age
  mileage_per_year
  make_category
  price_category


## 5. Verify Final Dataset

In [0]:
print(f"Final shape: {df.shape}")
print(f"\nNull check:")
total_nulls = df.isnull().sum().sum()
if total_nulls == 0:
    print("  No null values!")
else:
    print(df.isnull().sum()[df.isnull().sum() > 0].to_string())

print(f"\nData types:")
print(df.dtypes.to_string())

Final shape: (56133, 13)

Null check:
volume_cm3      47
drive_unit    1904
segment       5281

Data types:
priceUSD                int64
condition              object
mileage_kilometers    float64
fuel_type              object
volume_cm3            float64
color                  object
transmission           object
drive_unit             object
segment                object
car_age                 int64
mileage_per_year      float64
make_category          object
price_category         object


In [0]:
df.head(10)

,priceUSD,condition,mileage_kilometers,fuel_type,volume_cm3,color,transmission,drive_unit,segment,car_age,mileage_per_year,make_category,price_category
0,16800,with mileage,44000.0,petrol,2000.0,white,auto,front-wheel drive,j,3,14666.67,nissan,luxury
1,16900,with mileage,80000.0,petrol,1200.0,red,auto,front-wheel drive,j,3,26666.67,nissan,luxury
2,9650,with mileage,198000.0,petrol,1600.0,brown,mechanics,front-wheel drive,j,9,22000.00,nissan,luxury
3,9800,with mileage,117000.0,petrol,1600.0,black,mechanics,front-wheel drive,j,8,14625.00,nissan,luxury
4,11500,with mileage,93000.0,petrol,2000.0,other,auto,part-time four-wheel drive,j,8,11625.00,nissan,luxury
5,10900,with mileage,170000.0,petrol,2000.0,purple,auto,front-wheel drive,j,8,21250.00,nissan,luxury
6,15500,with mileage,124000.0,diesel,1500.0,black,mechanics,front-wheel drive,j,3,41333.33,nissan,luxury
7,9150,with mileage,102500.0,petrol,2000.0,gray,auto,front-wheel drive,j,10,10250.00,nissan,luxury
8,9999,with mileage,178000.0,diesel,1500.0,white,mechanics,front-wheel drive,j,9,19777.78,nissan,luxury
9,10000,with mileage,132000.0,petrol,1600.0,black,mechanics,front-wheel drive,j,7,18857.14,nissan,luxury


In [0]:
df.shape

(56133, 13)

## 6. Save to Catalog

In [0]:
# Convert to Spark and save as Delta table
df_spark = spark.createDataFrame(df)
df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.carprice_project_engineered_data")

# Verify
verify = spark.table("workspace.default.carprice_project_engineered_data").toPandas()
print(f"Saved carprice_project_engineered_data: ({len(verify)} rows, {len(verify.columns)} columns)")
print(f"\nColumns: {list(verify.columns)}")

Saved carprice_project_engineered_data: (56133 rows, 13 columns)

Columns: ['priceUSD', 'condition', 'mileage_kilometers', 'fuel_type', 'volume_cm3', 'color', 'transmission', 'drive_unit', 'segment', 'car_age', 'mileage_per_year', 'make_category', 'price_category']


### Feature Engineering Summary

| Action | Details |
|--------|---------|
| **Created** `car_age` | 2019 - year — age of car at time of listing |
| **Created** `mileage_per_year` | mileage_km / car_age — average annual usage intensity |
| **Created** `make_category` | Groups rare makes (< 100 cars) into 'other' |
| **Created** `price_category` | budget / mid / luxury bins based on 33rd & 66th percentiles — used for stratified split only, dropped before training |
| **Dropped** `model` | Very high cardinality — too sparse for encoding |
| **Dropped** `year` | Replaced by `car_age` |
| **Dropped** `make` | Replaced by `make_category` |